In [1]:
from pathlib import Path

print(Path.cwd())

/home/rupali-jha/solar-flare-watch/ml/notebooks


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project Root:", PROJECT_ROOT)

Project Root: /home/rupali-jha/solar-flare-watch/ml


In [3]:
from src.data_loader import load_events

In [4]:
from pathlib import Path

import pandas as pd

from src.data_loader import *
from src.preprocessing import *
from src.feature_engineering import *

In [5]:
# ---------------- Lightcurve ----------------

lightcurve_df = preprocess_lightcurve(
    load_lightcurve("czt1")
)

# ---------------- Spectra ----------------

metadata_df, counts_matrix, stat_err_matrix, channels = load_spectra("czt1")

metadata_df, counts_matrix = preprocess_spectra(
    metadata_df,
    counts_matrix
)

# ---------------- Housekeeping ----------------

hk_df = preprocess_housekeeping(
    load_housekeeping()
)

TypeError: preprocess_spectra() missing 1 required positional argument: 'observation_start'

In [6]:
lightcurve_features = engineer_lightcurve_features(
    lightcurve_df
)

lightcurve_features = engineer_peak_features(
    lightcurve_features
)

spectral_features = engineer_spectral_features(
    metadata_df,
    counts_matrix,
    channels
)

hk_features = engineer_housekeeping_features(
    hk_df
)

NameError: name 'hk_df' is not defined

In [7]:
print(lightcurve_features.shape)
print(spectral_features.shape)
print(hk_features.shape)

(43188, 42)
(2158, 21)


NameError: name 'hk_features' is not defined

In [8]:
hk_features["DATETIME"] = pd.to_datetime(
    hk_features["mjd"],
    unit="D",
    origin="1858-11-17"
)

NameError: name 'hk_features' is not defined

In [9]:
spectral_features["DATETIME"] = spectral_features["MID_TIME"]

KeyError: 'MID_TIME'

In [10]:
lightcurve_features = lightcurve_features.sort_values("DATETIME")

spectral_features = spectral_features.sort_values("DATETIME")

hk_features = hk_features.sort_values("DATETIME")

KeyError: 'DATETIME'

In [11]:
print(lightcurve_features["DATETIME"].min())
print(lightcurve_features["DATETIME"].max())

print()

print(spectral_features["DATETIME"].min())
print(spectral_features["DATETIME"].max())

print()

print(hk_features["DATETIME"].min())
print(hk_features["DATETIME"].max())

2026-06-24 12:00:01.739000
2026-06-24 23:59:48.739000



KeyError: 'DATETIME'

In [ ]:
print("Lightcurve :", lightcurve_features["DATETIME"].dtype)
print("Spectra    :", spectral_features["DATETIME"].dtype)
print("HK         :", hk_features["DATETIME"].dtype)

print(lightcurve_features["DATETIME"].head())
print(spectral_features["DATETIME"].head())
print(hk_features["DATETIME"].head())

Lightcurve : datetime64[us]
Spectra    : float64
HK         : datetime64[ns]
0   2026-06-24 12:00:01.739
1   2026-06-24 12:00:02.739
2   2026-06-24 12:00:03.739
3   2026-06-24 12:00:04.739
4   2026-06-24 12:00:05.739
Name: DATETIME, dtype: datetime64[us]
0    10.0
1    30.0
2    50.0
3    70.0
4    90.0
Name: DATETIME, dtype: float64
0   2026-06-24 12:00:01.239241541
1   2026-06-24 12:00:01.539189622
2   2026-06-24 12:00:09.193320434
3   2026-06-24 12:00:17.747234507
4   2026-06-24 12:00:25.401365942
Name: DATETIME, dtype: datetime64[ns]


In [12]:
spectral_features["DATETIME"] = pd.to_datetime(
    spectral_features["DATETIME"],
    unit="D",
    origin="1858-11-17"
)

KeyError: 'DATETIME'

In [13]:
spectral_features["DATETIME"] = pd.to_datetime(
    (
        spectral_features["TSTART"] +
        spectral_features["TSTOP"]
    ) / 2,
    unit="D",
    origin="1858-11-17"
)

In [14]:
hk_features["DATETIME"] = pd.to_datetime(
    hk_features["mjd"],
    unit="D",
    origin="1858-11-17"
)

NameError: name 'hk_features' is not defined

In [15]:
lightcurve_features["DATETIME"] = pd.to_datetime(
    lightcurve_features["DATETIME"]
).astype("datetime64[ns]")

spectral_features["DATETIME"] = pd.to_datetime(
    spectral_features["DATETIME"]
).astype("datetime64[ns]")

hk_features["DATETIME"] = pd.to_datetime(
    hk_features["DATETIME"]
).astype("datetime64[ns]")

NameError: name 'hk_features' is not defined

In [16]:
print(lightcurve_features["DATETIME"].dtype)
print(spectral_features["DATETIME"].dtype)
print(hk_features["DATETIME"].dtype)

datetime64[ns]
datetime64[ns]


NameError: name 'hk_features' is not defined

In [17]:
lightcurve_df = preprocess_lightcurve(load_lightcurve("czt1"))

metadata_df, counts_matrix, stat_err_matrix, channels = load_spectra("czt1")
metadata_df, counts_matrix = preprocess_spectra(
    metadata_df,
    counts_matrix
)

hk_df = preprocess_housekeeping(load_housekeeping())

TypeError: preprocess_spectra() missing 1 required positional argument: 'observation_start'

In [18]:
dataset = pd.merge_asof(
    lightcurve_features,
    spectral_features,
    on="DATETIME",
    direction="nearest",
    tolerance=pd.Timedelta("30s")
)

In [19]:
missing = (
    dataset
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing[missing > 0]

spec_mean_counts     43188
spec_std_counts      43188
spec_max             43188
spec_min             43188
peak_channel         43188
spectral_centroid    43188
spectral_spread      43188
spectral_entropy     43188
low_energy           43188
mid_energy           43188
active_channels      43188
high_energy          43188
dominant_fraction    43188
hardness_ratio2      43188
hardness_ratio1      43188
spec_total_counts    43188
TSTOP                43188
TSTART               43188
SPEC_NUM             43188
EXPOSURE             43188
ROWID                43188
dtype: int64

In [20]:
print(
    "Duplicate rows:",
    dataset.duplicated().sum()
)

Duplicate rows: 0


In [21]:
print(
    dataset["DATETIME"].is_monotonic_increasing
)

True


In [22]:
dataset.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 43188 entries, 0 to 43187
Data columns (total 63 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   MJD                   43188 non-null  float64       
 1   ISOT                  43188 non-null  str           
 2   COUNTS                43188 non-null  float64       
 3   STAT_ERR              43188 non-null  float64       
 4   DATETIME              43188 non-null  datetime64[ns]
 5   rolling_mean          43188 non-null  float64       
 6   rolling_std           43188 non-null  float64       
 7   rolling_max           43188 non-null  float64       
 8   rolling_min           43188 non-null  float64       
 9   ema_10                43188 non-null  float64       
 10  ema_30                43188 non-null  float64       
 11  diff1                 43188 non-null  float64       
 12  diff2                 43188 non-null  float64       
 13  gradient              43188

In [23]:
print(lightcurve_features["DATETIME"].min())
print(lightcurve_features["DATETIME"].max())

print()

print(spectral_features["DATETIME"].min())
print(spectral_features["DATETIME"].max())


print(lightcurve_features["DATETIME"].head())

print()

print(spectral_features["DATETIME"].head())


print(lightcurve_features["DATETIME"].iloc[0])

print(spectral_features["DATETIME"].iloc[0])

2026-06-24 12:00:01.739000
2026-06-24 23:59:48.739000

1858-11-27 00:00:00
1977-01-06 23:59:59.956799671
0   2026-06-24 12:00:01.739
1   2026-06-24 12:00:02.739
2   2026-06-24 12:00:03.739
3   2026-06-24 12:00:04.739
4   2026-06-24 12:00:05.739
Name: DATETIME, dtype: datetime64[ns]

0   1858-11-27
1   1858-12-17
2   1859-01-06
3   1859-01-26
4   1859-02-15
Name: DATETIME, dtype: datetime64[ns]
2026-06-24 12:00:01.739000
1858-11-27 00:00:00


In [24]:
print(metadata_df["TSTART"].head())

print(metadata_df["TSTOP"].head())
print(metadata_df["TSTART"].describe())

0     0.0
1    20.0
2    40.0
3    60.0
4    80.0
Name: TSTART, dtype: float64
0     20.0
1     40.0
2     60.0
3     80.0
4    100.0
Name: TSTOP, dtype: float64
count     2158.000000
mean     21570.000000
std      12462.105226
min          0.000000
25%      10785.000000
50%      21570.000000
75%      32355.000000
max      43140.000000
Name: TSTART, dtype: float64


In [25]:
observation_start = lightcurve_df["DATETIME"].min()

metadata_df["DATETIME"] = (
    observation_start +
    pd.to_timedelta(
        metadata_df["MID_TIME"],
        unit="s"
    )
)

KeyError: 'MID_TIME'

In [26]:
print(metadata_df["TSTART"].describe())
print(metadata_df["TSTOP"].describe())

count     2158.000000
mean     21570.000000
std      12462.105226
min          0.000000
25%      10785.000000
50%      21570.000000
75%      32355.000000
max      43140.000000
Name: TSTART, dtype: float64
count     2158.000000
mean     21590.000000
std      12462.105226
min         20.000000
25%      10805.000000
50%      21590.000000
75%      32375.000000
max      43159.999999
Name: TSTOP, dtype: float64


In [27]:
print(metadata_df["DATETIME"].head())

print(metadata_df["DATETIME"].tail())

KeyError: 'DATETIME'

In [28]:
spectral_features = engineer_spectral_features(
    metadata_df,
    counts_matrix,
    channels,
)

In [29]:
dataset = pd.merge_asof(
    lightcurve_features.sort_values("DATETIME"),
    spectral_features.sort_values("DATETIME"),
    on="DATETIME",
    direction="nearest",
    tolerance=pd.Timedelta("30s"),
)

KeyError: 'DATETIME'

In [30]:
dataset[
    [
        "spec_total_counts",
        "spectral_centroid",
        "hardness_ratio1",
    ]
].isna().sum()

spec_total_counts    43188
spectral_centroid    43188
hardness_ratio1      43188
dtype: int64

In [31]:
dataset = pd.merge_asof(
    dataset.sort_values("DATETIME"),
    hk_features.sort_values("DATETIME"),
    on="DATETIME",
    direction="nearest",
    tolerance=pd.Timedelta("30s"),
)

NameError: name 'hk_features' is not defined

In [32]:
missing = (
    dataset
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing[missing > 0]

spec_mean_counts     43188
spec_std_counts      43188
spec_max             43188
spec_min             43188
peak_channel         43188
spectral_centroid    43188
spectral_spread      43188
spectral_entropy     43188
low_energy           43188
mid_energy           43188
active_channels      43188
high_energy          43188
dominant_fraction    43188
hardness_ratio2      43188
hardness_ratio1      43188
spec_total_counts    43188
TSTOP                43188
TSTART               43188
SPEC_NUM             43188
EXPOSURE             43188
ROWID                43188
dtype: int64

In [33]:
print(dataset.shape)

print(dataset["DATETIME"].min())
print(dataset["DATETIME"].max())

print("Duplicate rows:", dataset.duplicated().sum())

print("Time sorted:", dataset["DATETIME"].is_monotonic_increasing)

(43188, 63)
2026-06-24 12:00:01.739000
2026-06-24 23:59:48.739000
Duplicate rows: 0
Time sorted: True


In [34]:
spectral_cols = spectral_features.columns.difference(["DATETIME"])

dataset[spectral_cols] = (
    dataset[spectral_cols]
    .ffill()
    .bfill()
)

In [35]:
dataset[spectral_cols].isna().sum().sum()

np.int64(906948)

In [36]:
print("=" * 60)
print("FINAL DATASET SUMMARY")
print("=" * 60)

print("Rows:", dataset.shape[0])
print("Columns:", dataset.shape[1])

print("\nTotal Missing Values:", dataset.isna().sum().sum())
print("Duplicate Rows:", dataset.duplicated().sum())
print("Time Sorted:", dataset["DATETIME"].is_monotonic_increasing)

print("\nTime Range")
print(dataset["DATETIME"].min())
print(dataset["DATETIME"].max())

FINAL DATASET SUMMARY
Rows: 43188
Columns: 63

Total Missing Values: 906948
Duplicate Rows: 0
Time Sorted: True

Time Range
2026-06-24 12:00:01.739000
2026-06-24 23:59:48.739000


In [37]:
from pathlib import Path

SAVE_DIR = Path("../data/processed")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

dataset.to_parquet(
    SAVE_DIR / "aligned_dataset.parquet",
    index=False,
)

dataset.to_csv(
    SAVE_DIR / "aligned_dataset.csv",
    index=False,
)

print("✅ aligned_dataset saved successfully!")

✅ aligned_dataset saved successfully!


In [38]:
import pandas as pd
import pyarrow as pa
import numpy as np

print("Pandas :", pd.__version__)
print("PyArrow:", pa.__version__)
print("NumPy  :", np.__version__)

Pandas : 3.0.4
PyArrow: 24.0.0
NumPy  : 2.5.0


In [39]:
print(dataset.dtypes)

MJD                         float64
ISOT                            str
COUNTS                      float64
STAT_ERR                    float64
DATETIME             datetime64[ns]
                          ...      
high_energy                 float64
hardness_ratio1             float64
hardness_ratio2             float64
dominant_fraction           float64
active_channels             float64
Length: 63, dtype: object


In [40]:
print(dataset.dtypes.value_counts())

float64           57
int64              3
str                2
datetime64[ns]     1
Name: count, dtype: int64


In [41]:
from pathlib import Path

SAVE_DIR = Path("../data/processed")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

dataset.to_csv(
    SAVE_DIR / "aligned_dataset.csv",
    index=False
)

print("CSV saved successfully.")

CSV saved successfully.


In [42]:
import pyarrow as pa

for col in dataset.columns:
    try:
        pa.array(dataset[col])
    except Exception as e:
        print(f"{col}: {type(e).__name__} -> {e}")

In [43]:
dataset.to_parquet(
    "../data/processed/aligned_dataset.parquet",
    engine="pyarrow",
    index=False
)

In [44]:
dataset.info()

print("\nShape:", dataset.shape)

print("\nMissing Values:", dataset.isna().sum().sum())

print("\nDuplicate Rows:", dataset.duplicated().sum())

print("\nTime Sorted:", dataset["DATETIME"].is_monotonic_increasing)

<class 'pandas.DataFrame'>
RangeIndex: 43188 entries, 0 to 43187
Data columns (total 63 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   MJD                   43188 non-null  float64       
 1   ISOT                  43188 non-null  str           
 2   COUNTS                43188 non-null  float64       
 3   STAT_ERR              43188 non-null  float64       
 4   DATETIME              43188 non-null  datetime64[ns]
 5   rolling_mean          43188 non-null  float64       
 6   rolling_std           43188 non-null  float64       
 7   rolling_max           43188 non-null  float64       
 8   rolling_min           43188 non-null  float64       
 9   ema_10                43188 non-null  float64       
 10  ema_30                43188 non-null  float64       
 11  diff1                 43188 non-null  float64       
 12  diff2                 43188 non-null  float64       
 13  gradient              43188